# Xeno-canto Gathering

Build a reproducible download set using XC query tags.


In [4]:
import json
import sys
import time
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from src.config import CONFIG
from src.dataset.utils.xeno_canto import write_raw_manifest_for_species, manifest_path
from src.dataset.utils.selection import write_selected_manifests


In [5]:
DATA_DIR = Path(CONFIG.paths.data_dir)
RAW_DIR = DATA_DIR / CONFIG.paths.raw_dir
MANIFEST_DIR = DATA_DIR / CONFIG.paths.manifests_dir

for p in [RAW_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    
SPECIES_FILE = Path("../../species_list_small.json")
species_map = json.loads(SPECIES_FILE.read_text(encoding="utf-8"))
species_list = list(species_map.values())
species_list[:5]

[{'common_name': 'European Robin', 'sci_name': 'Erithacus rubecula'},
 {'common_name': 'Eurasian Blackbird', 'sci_name': 'Turdus merula'},
 {'common_name': 'Eurasian Wren', 'sci_name': 'Troglodytes troglodytes'},
 {'common_name': 'Eurasian Blue Tit', 'sci_name': 'Cyanistes caeruleus'},
 {'common_name': 'Great Tit', 'sci_name': 'Parus major'}]

In [ ]:
def build_query(sci_name: str) -> str:
    xc = CONFIG.xeno_canto
    tags = [
        f'sp:"{sci_name}"',
        "grp:birds",
        "area:europe",
        'q:">C"',                 
        f'len:">{xc.min_len}"',
        f'len:"<{xc.max_len}"',
    ]
    return " ".join(tags)


## Fetch metadata and store in manifest files


In [7]:
for entry in species_list:
    sci_name = entry["sci_name"]
    query = build_query(sci_name)

    out_csv = write_raw_manifest_for_species(
        sci_name=sci_name,
        query=query,
        manifest_dir=MANIFEST_DIR,
        per_page=500,
    )

    print(f"Wrote raw manifest: {out_csv}")


Wrote raw manifest: bird_data\manifests\erithacus_rubecula.csv
Wrote raw manifest: bird_data\manifests\turdus_merula.csv
Wrote raw manifest: bird_data\manifests\troglodytes_troglodytes.csv
Wrote raw manifest: bird_data\manifests\cyanistes_caeruleus.csv
Wrote raw manifest: bird_data\manifests\parus_major.csv
Wrote raw manifest: bird_data\manifests\carduelis_carduelis.csv
Wrote raw manifest: bird_data\manifests\fringilla_coelebs.csv
Wrote raw manifest: bird_data\manifests\turdus_philomelos.csv
Wrote raw manifest: bird_data\manifests\pica_pica.csv
Wrote raw manifest: bird_data\manifests\coloeus_monedula.csv
Wrote raw manifest: bird_data\manifests\columba_livia.csv
Wrote raw manifest: bird_data\manifests\columba_palumbus.csv
Wrote raw manifest: bird_data\manifests\streptopelia_decaocto.csv
Wrote raw manifest: bird_data\manifests\sturnus_vulgaris.csv
Wrote raw manifest: bird_data\manifests\passer_domesticus.csv
Wrote raw manifest: bird_data\manifests\phylloscopus_trochilus.csv
Wrote raw man

In [ ]:
summary = write_selected_manifests(
    MANIFEST_DIR,
    target_per_species=250,             
    recordist_cap=30,
    prefer_countries=("Ireland", "United Kingdom"),
    selected_suffix="selected",
)

# Print a readable per-species summary
for fname, info in summary["per_file"].items():
    stats = info.get("stats", {})
    if not stats:
        print(fname, info)
        continue
    print(f"\n=== {fname} ===")
    print("selected:", stats["selected"], "/", stats["total_in"])
    print("IE/UK:", stats["preferred_selected"], f"({stats['preferred_selected_pct']:.1f}%)")
    print("month coverage:", stats["selected_month_coverage"])
    print("max by one recordist:", stats["max_selected_by_one_recordist"])
    print("top countries:", stats["selected_country_top"][:5])


=== accipiter_nisus.csv ===
selected: 124 / 124
IE/UK: 28 (22.6%)
month coverage: 9
max by one recordist: 10
top countries: [('United Kingdom', 24), ('Sweden', 20), ('France', 18), ('Poland', 17), ('Germany', 13)]

=== acrocephalus_schoenobaenus.csv ===
selected: 250 / 548
IE/UK: 124 (49.6%)
month coverage: 10
max by one recordist: 25
top countries: [('United Kingdom', 81), ('Ireland', 43), ('France', 32), ('Sweden', 23), ('Portugal', 14)]

=== aegithalos_caudatus.csv ===
selected: 250 / 704
IE/UK: 84 (33.6%)
month coverage: 12
max by one recordist: 27
top countries: [('United Kingdom', 68), ('Poland', 38), ('France', 27), ('Sweden', 22), ('Spain', 19)]

=== alcedo_atthis.csv ===
selected: 250 / 550
IE/UK: 83 (33.2%)
month coverage: 12
max by one recordist: 23
top countries: [('United Kingdom', 54), ('France', 44), ('Ireland', 29), ('Portugal', 29), ('Germany', 26)]

=== anthus_pratensis.csv ===
selected: 250 / 669
IE/UK: 124 (49.6%)
month coverage: 12
max by one recordist: 30
top cou